# Індивідуальні завдання до практичної роботи № 4
**Виконав:** Мельниківський Андрій, група КН-41

*Варіант не вказано, тому виконано всі завдання 1–7.*

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Підключення Google Диска (у Colab). Поза Colab — використовується локальна папка.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/'
except ImportError:
    BASE = ''
    print('Colab не виявлено — використовується локальна папка')
PATH = BASE + 'sales.csv'
print('Шлях до файлу:', PATH)

Colab не виявлено — використовується локальна папка
Шлях до файлу: sales.csv


In [3]:
df = pd.read_csv(PATH)
df['datetime'] = pd.to_datetime(df['datetime'])
df['day_of_week'] = df['datetime'].dt.dayofweek
df['hour'] = df['datetime'].dt.hour
df.head(3)

,date,datetime,cash_type,card,money,coffee_name,day_of_week,hour
0,2024-03-01,2024-03-01 10:19:37.400,card,ANON-0000-0000-0001,35.99,Cocoa,4,10
1,2024-03-01,2024-03-01 12:42:16.125,card,ANON-0000-0000-0002,26.88,Americano,4,12
2,2024-03-01,2024-03-01 13:16:18.819,card,ANON-0000-0000-0003,28.90,Americano,4,13


## Завдання 1. Кількість транзакцій за типом оплати (Series)

In [4]:
by_pay = df.groupby('cash_type').size()
print(by_pay)
print(type(by_pay))

cash_type
card    770
cash     30
dtype: int64
<class 'pandas.core.series.Series'>


## Завдання 2. `agg()`: максимальна та мінімальна сума за напоєм

In [5]:
df.groupby('coffee_name')['money'].agg(['max', 'min'])

,max,min
coffee_name,,
Americano,28.9,26.88
Americano with Milk,33.8,31.43
Cappuccino,38.7,35.99
Cocoa,38.7,35.99
Cortado,28.9,26.88
Espresso,24.0,22.32
Hot Chocolate,38.7,35.99
Latte,38.7,35.99


## Завдання 3. Зведена таблиця `cash_type × coffee_name` (сума `money`)

In [6]:
pd.pivot_table(df, values='money', index='cash_type', columns='coffee_name', aggfunc='sum', fill_value=0).round(2)

coffee_name,Americano,Americano with Milk,Cappuccino,Cocoa,Cortado,Espresso,Hot Chocolate,Latte
cash_type,,,,,,,,
card,2932.26,5924.59,4443.89,2314.63,1505.44,469.92,2358.75,6653.26
cash,53.76,164.26,265.48,74.69,57.80,0.00,116.10,340.17


## Завдання 4. Перехресна таблиця: день тижня × тип оплати

In [7]:
pd.crosstab(df['day_of_week'], df['cash_type'])

cash_type,card,cash
day_of_week,,
0,114,3
1,109,3
2,102,2
3,111,9
4,107,1
5,116,4
6,111,8


## Завдання 5. `merge()` із собівартістю та колонка `profit`

In [8]:
cost = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Hot Chocolate', 'Cappuccino', 'Americano with Milk', 'Espresso', 'Cocoa', 'Cortado'],
    'cost': [15, 10, 14, 14, 12, 8, 13, 11]
})
d5 = df.merge(cost, on='coffee_name', how='left')
d5['profit'] = d5['money'] - d5['cost']
print('Рядків без собівартості:', d5['cost'].isnull().sum())
d5[['coffee_name', 'money', 'cost', 'profit']].head(10)

Рядків без собівартості: 0


,coffee_name,money,cost,profit
0,Cocoa,35.99,13,22.99
1,Americano,26.88,10,16.88
2,Americano,28.90,10,18.90
3,Latte,35.99,15,20.99
4,Latte,35.99,15,20.99
5,Cortado,26.88,11,15.88
6,Americano,28.90,10,18.90
7,Americano,28.90,10,18.90
8,Cappuccino,38.70,14,24.70
9,Latte,38.70,15,23.70


In [9]:
print('Загальний прибуток:', d5['profit'].sum().round(2))
d5.groupby('coffee_name')['profit'].sum().round(2).sort_values(ascending=False)

Загальний прибуток: 17427.0


coffee_name
Latte                  4233.43
Americano with Milk    3880.85
Cappuccino             2973.37
Americano              1936.02
Cocoa                  1570.32
Hot Chocolate          1564.85
Cortado                 958.24
Espresso                309.92
Name: profit, dtype: float64

## Завдання 6. `concat()` двох штучних DataFrame

In [10]:
a = pd.DataFrame({'id': [1, 2, 3], 'name': ['Anna', 'Bohdan', 'Olena'], 'score': [85, 90, 78]}, index=[0, 1, 2])
b = pd.DataFrame({'id': [4, 5], 'name': ['Petro', 'Iryna'], 'score': [92, 88]}, index=[3, 4])
res = pd.concat([a, b])
print(res)
print('Індекси збережено:', list(res.index))

   id    name  score
0   1    Anna     85
1   2  Bohdan     90
2   3   Olena     78
3   4   Petro     92
4   5   Iryna     88
Індекси збережено: [0, 1, 2, 3, 4]


Індекси обох таблиць збережено. Якщо вони збігаються, у результаті з'являться дублікати індексу — тоді варто передати `ignore_index=True`:

In [11]:
c = pd.DataFrame({'id': [6], 'name': ['Oleh'], 'score': [70]})  # індекс 0 — збігається з a
print(pd.concat([a, c]), '\n')
print(pd.concat([a, c], ignore_index=True))

   id    name  score
0   1    Anna     85
1   2  Bohdan     90
2   3   Olena     78
0   6    Oleh     70 

   id    name  score
0   1    Anna     85
1   2  Bohdan     90
2   3   Olena     78
3   6    Oleh     70


## Завдання 7. Середній чек за годинами

In [12]:
avg_check = df.groupby('hour')['money'].mean().round(2)
print(avg_check)
print('\nНайвищий середній чек о', avg_check.idxmax(), 'год:', avg_check.max())

hour
8     34.75
9     34.48
10    34.15
11    33.77
12    34.91
13    34.47
14    34.62
15    35.59
16    33.84
17    34.58
18    34.98
19    34.98
20    34.50
Name: money, dtype: float64

Найвищий середній чек о 15 год: 35.59


## Відповіді на питання для самоперевірки

**1. Призначення `groupby()`.** Розділяє DataFrame на групи за значеннями колонки (колонок). Сам по собі обчислень не виконує — створює об'єкт `GroupBy`; розрахунок відбувається після застосування агрегатної функції.

**2. Кілька агрегатних функцій.** `df.groupby('coffee_name')['money'].agg(['sum', 'mean', 'count'])`.

**3. `size()` проти `count()`.** `size()` повертає кількість рядків у групі (разом із пропусками); `count()` — кількість непорожніх значень у кожній колонці. Приклад — нижче.

**4. Мультиіндекс → звичайний DataFrame.** `reset_index()`, для `Series` — `reset_index(name='count')`.

**5. Зведена таблиця.** Метод `pd.pivot_table()`. `values` — що агрегуємо, `index` — що стає рядками, `columns` — що стає стовпцями, `aggfunc` — функція агрегації (`sum`, `mean`, `count`…).

**6. Перехресна таблиця.** `pd.crosstab()` підраховує частоти комбінацій двох категоріальних змінних. Відмінність від `pivot_table`: за замовчуванням рахує кількість спостережень, не потребує числових `values`.

**7. Три способи об'єднання.** `concat` — склеювання таблиць «одна під одною» чи «поряд»; `merge` — з'єднання за значеннями спільних колонок (як SQL JOIN); `join` — об'єднання за індексами.

**8. Параметр `how` у `merge()`.** `inner` — лише спільні ключі; `left` — усі рядки лівої таблиці; `right` — усі рядки правої; `outer` — усі рядки обох; `cross` — декартів добуток.

**9. Коли `concat`.** Коли таблиці мають однакову структуру і їх треба просто з'єднати (додати рядки або стовпці) без зіставлення за ключем.

**10. Частка групи від загальної суми.** 1) знайти загальну суму `total = df['money'].sum()`; 2) обчислити суму за групами `df.groupby('coffee_name')['money'].sum()`; 3) поділити на `total` і помножити на 100.

**11. Колонка з днями тижня.** `df['day_of_week'] = pd.to_datetime(df['datetime']).dt.dayofweek`. Це дозволяє групувати та порівнювати продажі за днями тижня (сезонність).

**12. Збереження результатів агрегації.** `result.to_csv('file.csv')` зберігає індекс як колонку(и) (для мультиіндексу — кілька колонок); або `result.reset_index().to_csv('file.csv', index=False)` для плоскої таблиці.

In [13]:
# Демонстрація до питання 3: size() проти count() (колонка card має пропуски для оплати готівкою)
g = df.groupby('cash_type')
print('size():'); print(g.size(), '\n')
print('count() по колонці card:'); print(g['card'].count())

size():
cash_type
card    770
cash     30
dtype: int64 

count() по колонці card:
cash_type
card    770
cash      0
Name: card, dtype: int64
